# Kaiku Finance - agent test (local)

Tests the Kaiku agent: a finance coach that answers from three sources and labels each one.

| Source | When | Label |
|---|---|---|
| Model training | timeless, non-local concepts | `[MODEL]` |
| Exa (live web) | facts that change, depend on place, are niche, or go into a letter | `[WEB n]` |
| User transactions | the user's own money | `[DATA]` |

Needs a `.env` file in the project root with `ANTHROPIC_API_KEY` and `EXA_API_KEY`
(redeem `EXA50JUNCTION` at dashboard.exa.ai/billing). Every cell below calls paid APIs.

In [ ]:
# %pip install -r ../requirements.txt
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook/ -> ../src

import pandas as pd
from project_aethon.kaiku_agent import (MODEL_NAME, build_system_prompt, create_agent, format_results,
                         load_transactions, make_data_tool, make_exa_tool, run_agent, search_exa)
from project_aethon.kaiku_eval import (EXA, DATA, QUESTIONS, ask, check, compare_exa_configs, label_report,
                        routing_eval, run_variants, schema_tokens, show, token_report,
                        unanswerable_check)

pd.set_option("display.max_colwidth", 120)
transactions = load_transactions()   # synthetic sample; pass a CSV path for the S-Bank dataset
agent = create_agent(transactions=transactions)
transactions.tail()

## 1. One question per route

Expected: no tools for the concept, `exa_search` for the live fact, `query_spending` for the user's money.

In [ ]:
s_model = check(agent, "What is a variable interest rate?", set())
s_web = check(agent, "What is the 12-month Euribor today?", {EXA})
s_data = check(agent, "How much did I spend on taxis last month?", {DATA})

In [ ]:
for s in (s_model, s_web, s_data):
    show(s)
    print("labels:", label_report(s), "\n" + "-" * 80)

## 2. The action case: a letter must cite sources

A letter goes through Exa even if the model thinks it knows the rule.

In [ ]:
s_letter = check(agent, "My landlord in Helsinki raised my rent by 8% in the middle of the contract. "
                        "Is that legal? Draft a short letter to object.", {EXA})
show(s_letter)
label_report(s_letter)

## 3. What the tool costs in tokens

`toolmsg_billed_tokens` counts each search result once per LLM call that re-sends it.

In [ ]:
rep_web = token_report(s_web)

In [ ]:
# Fixed cost of the tool schemas on every call (system prompt not included)
from langchain.chat_models import init_chat_model
schema_tokens(init_chat_model(MODEL_NAME, temperature=0),
              [make_data_tool(transactions), make_exa_tool()])

In [ ]:
# Size of the system prompt (chars / 4 ~ tokens). It is re-sent on every call.
len(build_system_prompt()) // 4

## 4. What Exa returns

Raw normalised results, then the compact text the model actually reads.

In [ ]:
raw = search_exa("Kela general housing allowance amount", num_results=3, highlight_chars=500)
print("Exa cost (USD):", raw["cost"])
print(format_results(raw["results"], 500))

## 5. Exa settings: tokens vs hit rate

Highlights against full text, number of results, official domains only, and the `lite` search type.
`hit_rate` uses the expected texts in `EXA_EVAL`; check those keys before trusting it.

In [ ]:
cfg = compare_exa_configs()
cfg

## 6. Routing over the full question set

`route_ok` = used exactly the expected tools. `label_problems` lists wrong or missing labels and
any URL that does not come from the search results.

In [ ]:
routing = routing_eval(agent, QUESTIONS, repeats=1)
print("route_ok:", routing["route_ok"].mean().round(2),
      "| answers without label problems:", (routing["label_problems"] == "").mean().round(2))
routing

## 7. When nothing reliable exists

The agent should search, say it found nothing, and not fill the gap from memory. Read the answers.

In [ ]:
unanswerable_check(agent)

## 8. Agent variants: cheaper without losing quality?

Same questions, different `exa_search` settings. Pick the cheapest variant that keeps `route_ok` and `label_ok`.

In [ ]:
def variant(**kw):
    return create_agent(transactions=transactions, exa_tool=make_exa_tool(**kw))

variants = {
    "default (3 results, 500 chars)": agent,
    "2 results, 300 chars": variant(num_results=2, highlight_chars=300),
    "5 results, 500 chars": variant(num_results=5, highlight_chars=500),
    "full text 3000 chars": variant(num_results=3, full_text_chars=3000),
}
var = run_variants(variants, QUESTIONS, repeats=1)
var

## 9. Try your own question

In [ ]:
s = ask(agent, "What happens to my budget if I stop ordering food delivery?")
show(s)
token_report(s);